# Train on Colab GPU (reproducible)

This notebook does **not** use Google Drive. It clones the repo at a commit, pulls the exact
dataset version recorded in `dvc.lock` from the DVC remote, trains, and pushes the model back.
Any result can be traced to a Git commit and a dataset hash.

**Before running:** Runtime → Change runtime type → GPU. Add these in Colab *Secrets* (key icon):
`DVC_ACCESS_KEY_ID`, `DVC_SECRET_ACCESS_KEY` (your R2/S3 remote), optionally
`MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (e.g. DagsHub),
and `GITHUB_TOKEN` if you want to push `dvc.lock` back from here.

In [ ]:
REPO = "https://github.com/<your-user>/av-sensor-dataops.git"   # <- change
REF = "main"                                                      # branch, tag or commit

In [ ]:
!git clone -q $REPO repo
%cd repo
!git checkout -q $REF && git log --oneline -1
!pip install -q -e ".[train,dvc]"

In [ ]:
import os
from google.colab import userdata

for env, key in {"AWS_ACCESS_KEY_ID": "DVC_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY": "DVC_SECRET_ACCESS_KEY",
                 "MLFLOW_TRACKING_URI": "MLFLOW_TRACKING_URI",
                 "MLFLOW_TRACKING_USERNAME": "MLFLOW_TRACKING_USERNAME",
                 "MLFLOW_TRACKING_PASSWORD": "MLFLOW_TRACKING_PASSWORD"}.items():
    try:
        os.environ[env] = userdata.get(key)
    except Exception:
        print("not set:", key)

## Get exactly the dataset version this commit points to

In [ ]:
!dvc pull data/ml_ready
!cat data/ml_ready/yolo/manifest.json

## Train, evaluate, export (DVC skips stages whose inputs did not change)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!dvc repro export_onnx

In [ ]:
!dvc metrics show
!cat models/model_card.json

## Save the results
`dvc push` uploads the model to the remote. Then commit `dvc.lock` + metrics so the
training run is part of the project history (here, or download `dvc.lock` and commit locally).

In [ ]:
!dvc push
!git status --short

In [ ]:
# optional: push the lineage record from Colab
# token = userdata.get("GITHUB_TOKEN")
# !git config user.email "you@example.com" && git config user.name "Your Name"
# !git add dvc.lock metrics models/model_card.json && git commit -m "train on Colab GPU"
# !git push https://$token@github.com/<your-user>/av-sensor-dataops.git HEAD:$REF